In [1]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt

DATASET_FOLDER = "/kaggle/input/uw-madison-gi-tract-image-segmentation"


In [2]:
df_train = pd.read_csv(os.path.join(DATASET_FOLDER, "train.csv"))
print(f"size: {len(df_train)}")
display(df_train.head())


size: 95088


,id,class,segmentation
0,case77_day20_slice_0001,large_bowel,NaN
1,case77_day20_slice_0001,small_bowel,NaN
2,case77_day20_slice_0001,stomach,NaN
3,case77_day20_slice_0002,large_bowel,NaN
4,case77_day20_slice_0002,small_bowel,NaN


In [3]:
df_ssub = pd.read_csv(os.path.join(DATASET_FOLDER, "sample_submission.csv"))
print(f"size: {len(df_ssub)}")
display(df_ssub.head())


size: 20400


,id,class,predicted
0,case123_day20_slice_0001,large_bowel,1 1 5 2
1,case123_day20_slice_0001,small_bowel,1 1 5 2
2,case123_day20_slice_0001,stomach,1 1 5 2
3,case123_day20_slice_0002,large_bowel,1 1 5 2
4,case123_day20_slice_0002,small_bowel,1 1 5 2


In [4]:
ex_ids = [i not in df_ssub["id"] for i in df_train["id"]]
print(f"missing: {len(ex_ids)}")


missing: 95088


In [5]:
# Switch between parsing test or train folder
PATTERN = ("case*", "case*_day*", "scans", "*.png")
if df_ssub.empty:
    print("using TRAIN dataset?")
    all_imgs = glob.glob(os.path.join(DATASET_FOLDER, "train", *PATTERN))
else:
    print("using TEST dataset!")
    all_imgs = glob.glob(os.path.join(DATASET_FOLDER, "test", *PATTERN))

print(f"images: {len(all_imgs)}")
labels = sorted(df_train["class"].unique())
print(f"labels: {labels}")


using TEST dataset!
images: 6800
labels: ['large_bowel', 'small_bowel', 'stomach']


In [6]:
def extract_details(ip):
    img = os.path.basename(ip)
    im_name, _ = os.path.splitext(img)
    folders = ip.split(os.path.sep)
    case_day = folders[-3].split("_")
    case = case_day[0].replace("case", "")
    day = case_day[1].replace("day", "")
    slice_id = im_name.split("_")[1]
    img_path = img.replace(DATASET_FOLDER + "/", "")
    return {
        "id": f"case{case}_day{day}_slice_{slice_id}",
        "Case": int(case),
        "Day": int(day),
        "Slice": slice_id,
        "image": img,
        "image_path": img_path,
    }

extract_details(all_imgs[0])


{'id': 'case89_day21_slice_0024',
 'Case': 89,
 'Day': 21,
 'Slice': '0024',
 'image': 'slice_0024_276_276_1.63_1.63.png',
 'image_path': 'slice_0024_276_276_1.63_1.63.png'}

In [7]:
from tqdm.auto import tqdm

df = pd.DataFrame([extract_details(ip) for ip in tqdm(all_imgs)])
print(f"size: {len(df)}")
display(df.head(3))


  0%|          | 0/6800 [00:00<?, ?it/s]

size: 6800


,id,Case,Day,Slice,image,image_path
0,case89_day21_slice_0024,89,21,0024,slice_0024_276_276_1.63_1.63.png,slice_0024_276_276_1.63_1.63.png
1,case89_day21_slice_0013,89,21,0013,slice_0013_276_276_1.63_1.63.png,slice_0013_276_276_1.63_1.63.png
2,case89_day21_slice_0015,89,21,0015,slice_0015_276_276_1.63_1.63.png,slice_0015_276_276_1.63_1.63.png


In [8]:
ex_ids = [i not in df_ssub["id"] for i in df["id"]]
print(f"missing: {len(ex_ids)}")


missing: 6800


In [9]:
sub = [{"id": row["id"], "class": lb, "predicted": f"{i} 1"}
       for _, row in tqdm(df.iterrows(), total=len(df)) for i, lb in enumerate(labels)]
df_sub = pd.DataFrame(sub)
display(df_sub.head())


  0%|          | 0/6800 [00:00<?, ?it/s]

,id,class,predicted
0,case89_day21_slice_0024,large_bowel,0 1
1,case89_day21_slice_0024,small_bowel,1 1
2,case89_day21_slice_0024,stomach,2 1
3,case89_day21_slice_0013,large_bowel,0 1
4,case89_day21_slice_0013,small_bowel,1 1


In [10]:
df_keys = [f"{rr['id']}-//-{rr['class']}" for _, rr in tqdm(df_sub.iterrows())]
if not df_ssub.empty:
    ss_keys = [f"{rr['id']}-//-{rr['class']}" for _, rr in tqdm(df_ssub.iterrows())]
    assert len(df_sub) == len(df_ssub)
    miss = [k for k in df_keys if k not in ss_keys]
    assert not miss


0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [11]:
del df_ssub['predicted']
df_sub = df_ssub.merge(df_sub, on=['id','class'])

df_sub[['id', 'class', 'predicted']].to_csv("submission.csv", index=False)

!head submission.csv


id,class,predicted
case123_day20_slice_0001,large_bowel,0 1
case123_day20_slice_0001,small_bowel,1 1
case123_day20_slice_0001,stomach,2 1
case123_day20_slice_0002,large_bowel,0 1
case123_day20_slice_0002,small_bowel,1 1
case123_day20_slice_0002,stomach,2 1
case123_day20_slice_0003,large_bowel,0 1
case123_day20_slice_0003,small_bowel,1 1
case123_day20_slice_0003,stomach,2 1
